# Stack Overflow Q&A in Relate — Tags, Co-occurrence, and Visualization

This notebook loads the **latest 100 questions and answers** for a Stack Overflow tag
of your choice (default: `neo4j`) into an in-memory
Relate graph, runs the exploration
queries from the
[Stack Overflow graph example](https://github.com/neo4j-graph-examples/stackoverflow/blob/main/documentation/stackoverflow.adoc),
correlates tags the way the
[TagOverflow](https://medium.com/data-science/tagoverflow-correlating-tags-in-stackoverflow-66e2b0e1117b)
article does — pairwise co-occurrence and Jaccard similarity — and renders the
resulting tag graph with the
[neo4j-viz](https://neo4j.com/docs/python-graph-visualization/current/) library.

The graph model follows the Stack Overflow example (comments omitted — this
notebook loads questions and answers):

```text
(:User)-[:ASKED]->(:Question)
(:User)-[:PROVIDED]->(:Answer)
(:Answer)-[:ANSWERED]->(:Question)
(:Question)-[:TAGGED]->(:Tag)
```

> **Note** — this notebook needs network access to the public Stack Exchange API
> and an interactive visualization stack, so unlike the notebooks directly under
> `examples/python/` it is not executed by CI (that job globs only that directory)
> and is validated manually.

In [1]:
# --- Setup ------------------------------------------------------------------
# Install Relate and this notebook's dependencies into the current kernel
# if they are not already present. Relate is not yet on PyPI, so it is
# built from this repository checkout (requires the Rust toolchain,
# https://rustup.rs) -- the same maturin build that `make install-python` runs.
# Once published, replace the local-build line with:
#   pip install relate
import subprocess
import sys


def _ensure(module, pip_spec):
    try:
        __import__(module)
    except ImportError:
        print(f"Installing {pip_spec} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pip_spec.split()])


_ensure("relate", "../../../surfaces/relate-python")  # local wheel (pre-PyPI)
# _ensure("relate", "relate")  # published-wheel placeholder
# 1.12 drops Python 3.10 (Requires-Python >= 3.11); the old [notebook]
# extra is gone (the Jupyter deps are core deps since 1.11).
_ensure("neo4j_viz", "neo4j-viz>=1.12.0")  # interactive graph visualization

import relate.viz

print("relate", getattr(relate, "__version__", "local build"))

relate 1.4.1


---
## 1. Configure the tag
Any single Stack Overflow tag works — `neo4j`, `cypher`, `rust`, `pytorch`, ...

In [2]:
TAG = "neo4j"   # the Stack Overflow tag to explore
PAGESIZE = 100  # how many of the latest questions to load

assert TAG.replace("-", "").isalnum(), "a single simple tag, please"

---
## 2. Fetch the latest questions and answers
The public [Stack Exchange API](https://api.stackexchange.com/docs/questions)
returns questions with their tags, owners, and answers embedded — the same filter
the Stack Overflow graph example uses for its `apoc.load.json` import.

In [3]:
import json
import urllib.request

API = (
    "https://api.stackexchange.com/2.3/questions"
    "?pagesize={pagesize}&order=desc&sort=creation&tagged={tag}&site=stackoverflow"
    "&filter=!5-i6Zw8Y)4W7vpy91PMYsKM-k9yzEsSC1_Uxlf"
)

with urllib.request.urlopen(API.format(pagesize=PAGESIZE, tag=TAG), timeout=30) as r:
    data = json.loads(r.read().decode("utf-8"))

questions = data["items"]
answers = [a for q in questions for a in q.get("answers", [])]
print(len(questions), "questions,", len(answers), "answers fetched for tag", repr(TAG))
print("API quota remaining:", data.get("quota_remaining"))

100 questions, 99 answers fetched for tag 'neo4j'
API quota remaining: 297


In [4]:
import time

# Set True to also echo each statement's raw rows (each cell's return value);
# the printed tables are the readable form and stay.
SHOW_JSON = False

db = relate.connect(":memory:")


def show(title, rows, elapsed_ms, stats):
    """Print rows as a plain table with its timing and mutation counts."""
    print(f"\n{title} ({len(rows)} rows, {elapsed_ms:.1f} ms)")
    if rows:
        cols = list(rows[0].keys())
        widths = [max(len(c), *(len(str(r[c])) for r in rows)) for c in cols]
        print("  " + "  ".join(c.ljust(w) for c, w in zip(cols, widths)))
        print("  " + "  ".join("-" * w for w in widths))
        for r in rows:
            print("  " + "  ".join(str(r[c]).ljust(w) for c, w in zip(cols, widths)))

    # Mutation counts come back from the engine's stats (the same counters
    # the CLI reports); a read statement changes nothing and prints none.
    changed = {name: value for name, value in stats.items() if value}
    if changed:
        summary = ", ".join(
            f"{value} {name.replace('_', ' ')}"
            for name, value in sorted(changed.items())
        )
        print(f"  changed: {summary}")


def run(title, query, params=None):
    """Execute one query and show its rows, its timing, and what it changed."""
    cursor = db.execute(query, params or {})
    rows = cursor.fetchall()
    summary = cursor.consume()  # counters, query type, elapsed_ms
    show(title, rows, summary.elapsed_ms, summary.counters)
    return rows if SHOW_JSON else None

---
## 3. Declare the schema — a graph type
The data-derived schema comes later (section 5); a **graph type** declares the
schema up front (GQL's schema DDL), with typed properties. The `=>`
marker after a label denotes the *identifying* label, and each property is
declared as `name IS <value type>` (the GQL spelling of `name :: <value type>`);
`IS KEY` / `IS UNIQUE` on a property are its identity constraints.

Deliberate omissions show that a graph type is *open* — it constrains only what
it declares. `User.uuid` stays undeclared (deleted accounts make it
int-or-string); any property not listed here (scores' up/down breakdown,
profile images, ...) is unconstrained; and `Question.title` stays undeclared
because the full-text index in section 6 is what serves title lookups.

The declared type first lands in the catalog; promoting it with
`ALTER CURRENT GRAPH TYPE SET GRAPH TYPE <name>` makes it the database's
current type — and derives the enforcing constraints from it.

In [5]:
run("declare the graph type", """
    CREATE GRAPH TYPE SOGraph AS {
      (:Question => {uuid :: INTEGER NOT NULL IS KEY, score :: INTEGER,
                     view_count :: INTEGER, answer_count :: INTEGER,
                     is_answered :: BOOLEAN, link :: STRING}),
      (:Answer => {uuid :: INTEGER NOT NULL IS KEY, score :: INTEGER,
                   is_accepted :: BOOLEAN, link :: STRING}),
      (:User => {display_name :: STRING, reputation :: INTEGER,
                 user_type :: STRING, link :: STRING}),
      (:Tag => {name :: STRING NOT NULL IS UNIQUE}),
      (:User)-[:ASKED]->(:Question),
      (:Question)-[:TAGGED]->(:Tag),
      (:Answer)-[:ANSWERED]->(:Question),
      (:User)-[:PROVIDED]->(:Answer)
    }
    """)

run("promote it to the current graph type",
    "ALTER CURRENT GRAPH TYPE SET GRAPH TYPE SOGraph")

run("the current graph type, canonical form", "SHOW CURRENT GRAPH TYPE")


declare the graph type (0 rows, 15.6 ms)

promote it to the current graph type (0 rows, 4.3 ms)
  changed: 29 constraints added

the current graph type, canonical form (1 rows, 1.2 ms)
  specification                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [6]:
# Promotion derives the enforcing constraints: key and uniqueness on the
# identifying properties, a property-type constraint per declared property,
# and source/target label constraints per declared relationship type. The
# summary shows the kinds; SHOW CONSTRAINTS lists them all individually.
run("constraints derived from the graph type (by kind)", """
    SHOW CONSTRAINTS
    YIELD type
    RETURN type, count(*) AS constraints
    ORDER BY constraints DESC, type
    """)


constraints derived from the graph type (by kind) (6 rows, 2.3 ms)
  type                       constraints
  -------------------------  -----------
  NODE_PROPERTY_TYPE         15         
  RELATIONSHIP_SOURCE_LABEL  4          
  RELATIONSHIP_TARGET_LABEL  4          
  NODE_PROPERTY_EXISTENCE    3          
  NODE_KEY                   2          
  NODE_PROPERTY_UNIQUENESS   1          


In [7]:
# The promoted type validates writes from the very next statement on — even
# before the import. A mistyped property and a duplicate key are both rejected,
# with the constraint's name in the error.
db.execute("CREATE (:Question {uuid: -1, score: 0})")  # well-typed, own row
for label, stmt in [
    ("mistyped score", "CREATE (:Question {uuid: -2, score: 'not-a-number'})"),
    ("duplicate key", "CREATE (:Question {uuid: -1, score: 0})"),
]:
    try:
        db.execute(stmt)
        print(f"{label}: unexpectedly accepted")
    except relate.Error as e:
        print(f"{label} rejected: {e}")

# Clean the demo row up again: the import follows.
db.execute("MATCH (q:Question {uuid: -1}) DELETE q")
print("well-typed write accepted and cleaned up")

mistyped score rejected: semantic error: ConstraintValidationFailed: `constraint_73636fc3` property type
duplicate key rejected: semantic error: ConstraintValidationFailed: `constraint_233a8716` uniqueness
well-typed write accepted and cleaned up


The current type also renders as a virtual graph — `SHOW CURRENT GRAPH TYPE
AS GRAPH` returns one node per element type and one edge per declared
relationship type, ready for the same visualization helper:

![graph type](assets/stackoverflow_graph_type.png)

In [8]:
graph_type = relate.viz.to_visualization_graph(
    db.execute("SHOW CURRENT GRAPH TYPE AS GRAPH"),
    node_caption={"NodeElementType": "label"},
)

from neo4j_viz.colors import NEO4J_COLORS_DISCRETE, ColorSpace

# The synthetic nodes carry the element label under `label`; the edges carry
# the declared type under `relationshipType` — surface both as captions.
for n in graph_type.nodes:
    n.size = 14
# Color by the element type (Question/Tag/Answer/User), like section 5's
# data-derived schema, not by the synthetic NodeElementType label.
graph_type.color_nodes(property="label", color_space=ColorSpace.DISCRETE,
                       colors=NEO4J_COLORS_DISCRETE)
for r in graph_type.relationships:
    r.caption = r.properties["relationshipType"]

relate.viz.render_html(graph_type, initial_zoom=2)

---
## 4. Load the graph into Relate
Each `UNWIND` + `MERGE` statement below mirrors one part of the Stack Overflow
example's import statement — idempotent, so re-running the notebook with newer
questions merges into the same entities instead of duplicating them.

The API response carries more than the example uses, and the load keeps the
interesting parts: scores, up/down votes, view/favorite/comment counts and
last-activity on questions and answers, links everywhere, and on users the
reputation (karma), accept rate, user type and profile image. The full question/
answer bodies (`body_markdown`) are one commented line each — uncomment to store
them. Bios (`about_me`) need a separate `/users` call; a ready-to-uncomment cell
follows the load.

The `run` helper (defined above) executes a statement and prints its rows, its
timing and what it changed as a plain table.

In [9]:
run("questions", """
    UNWIND $questions AS q
    MERGE (question:Question {uuid: q.question_id})
      ON CREATE SET question.title = q.title,
        question.creation_date = q.creation_date,
        question.score = q.score,
        question.up_vote_count = q.up_vote_count,
        question.down_vote_count = q.down_vote_count,
        question.view_count = q.view_count,
        question.favorite_count = q.favorite_count,
        question.comment_count = q.comment_count,
        question.answer_count = q.answer_count,
        question.is_answered = q.is_answered,
        question.last_activity_date = q.last_activity_date,
        question.link = q.link
        // , question.body_markdown = q.body_markdown,  // uncomment to store full text
    """, {"questions": questions})

run("users + ASKED", """
    UNWIND $questions AS q
    MERGE (owner:User {uuid: coalesce(q.owner.user_id, 'deleted')})
      ON CREATE SET owner.display_name = coalesce(q.owner.display_name, 'deleted'),
        owner.reputation = q.owner.reputation,
        owner.accept_rate = q.owner.accept_rate,
        owner.user_type = q.owner.user_type,
        owner.profile_image = q.owner.profile_image,
        owner.link = q.owner.link
    MATCH (question:Question {uuid: q.question_id})
    MERGE (owner)-[:ASKED]->(question)
    """, {"questions": questions})

run("tags + TAGGED", """
    UNWIND $questions AS q
    MATCH (question:Question {uuid: q.question_id})
    UNWIND q.tags AS tagName
    MERGE (tag:Tag {name: tagName})
    MERGE (question)-[:TAGGED]->(tag)
    """, {"questions": questions})

run("answers + ANSWERED + PROVIDED", """
    UNWIND $questions AS q
    MATCH (question:Question {uuid: q.question_id})
    UNWIND q.answers AS a
    MERGE (answer:Answer {uuid: a.answer_id})
      ON CREATE SET answer.score = a.score,
        answer.up_vote_count = a.up_vote_count,
        answer.down_vote_count = a.down_vote_count,
        answer.comment_count = a.comment_count,
        answer.is_accepted = a.is_accepted,
        answer.creation_date = a.creation_date,
        answer.last_activity_date = a.last_activity_date,
        answer.link = a.link
        // , answer.body_markdown = a.body_markdown,  // uncomment to store full text
    MERGE (answer)-[:ANSWERED]->(question)
    MERGE (answerer:User {uuid: coalesce(a.owner.user_id, 'deleted')})
      ON CREATE SET answerer.display_name = coalesce(a.owner.display_name, 'deleted'),
        answerer.reputation = a.owner.reputation,
        answerer.accept_rate = a.owner.accept_rate,
        answerer.user_type = a.owner.user_type,
        answerer.profile_image = a.owner.profile_image,
        answerer.link = a.owner.link
    MERGE (answerer)-[:PROVIDED]->(answer)
    """, {"questions": questions})


questions (0 rows, 4.0 ms)
  changed: 100 labels added, 100 nodes created, 1300 properties set

users + ASKED (0 rows, 3.0 ms)
  changed: 87 labels added, 87 nodes created, 541 properties set, 100 relationships created

tags + TAGGED (0 rows, 3.1 ms)
  changed: 96 labels added, 96 nodes created, 96 properties set, 293 relationships created

answers + ANSWERED + PROVIDED (0 rows, 4.8 ms)
  changed: 134 labels added, 134 nodes created, 1106 properties set, 198 relationships created


In [10]:
# Optional: user bios. `about_me` is not in the questions filter, so fetch it
# from /users (at most 100 ids per call) with a filter that includes it —
# build one at https://api.stackexchange.com/docs/filters — and merge it on:
#
# uuids = [
#     r["uuid"]
#     for r in db.execute("MATCH (u:User) RETURN u.uuid AS uuid").fetchall()
#     if r["uuid"] != "deleted"
# ]
# bios = []
# for i in range(0, len(uuids), 100):
#     url = "https://api.stackexchange.com/2.3/users/" + ";".join(map(str, uuids[i : i + 100]))
#     url += "?site=stackoverflow&filter=<YOUR-FILTER-INCLUDING-about_me>"
#     with urllib.request.urlopen(url, timeout=30) as r:
#         bios.extend(json.loads(r.read().decode("utf-8"))["items"])
# db.execute(
#     "UNWIND $users AS u MATCH (o:User {uuid: u.user_id}) SET o.about_me = u.about_me",
#     {"users": bios},
# )
# print(f"{len(bios)} user bios loaded")

---
## 5. The schema at a glance
Before querying, look at what was loaded: `CALL db.schema.visualization` derives
the schema from the data itself — one node per label, one edge per relationship
type — and returns it as graph elements, so it feeds straight into the
visualization helper. Nothing here is hard-coded, so the picture follows whatever
the API returned.

Static preview (GitHub strips the interactive HTML below):

![schema](assets/stackoverflow_schema.png)

In [11]:
schema = relate.viz.to_visualization_graph(db.execute("CALL db.schema.visualization"))

# Size each label node by its entity count, area-proportional (radius ~
# sqrt(value)): neo4j-viz's `size` is linear (diameter), which over-emphasizes
# big counts — the circle's area would grow quadratically.
import math

def area_size(value, scale=6.0, base=8):
    """Radius such that the circle area grows linearly with the value."""
    return base + scale * math.sqrt(max(value, 0))


def size_nodes(viz, value_of, *, scale=6.0, base=8):
    """Size each node area-proportionally to value_of(node)."""
    for n in viz.nodes:
        n.size = area_size(value_of(n), scale, base)


entity_counts = {
    r["label"]: r["entities"]
    for r in db.execute("""
        MATCH (n)
        WITH labels(n)[0] AS label, count(*) AS entities
        RETURN label, entities
        """).fetchall()
}
size_nodes(schema, lambda n: entity_counts.get(n.properties["labels"][0], 1),
            scale=1.2, base=8)
relate.viz.render_html(schema, initial_zoom=1)

---
## 6. Search — full-text and vector indexes
The graph type already indexed `Question.uuid` (its key constraint derives the
index behind the load's MERGEs). Beyond that, Relate has two index kinds
that answer questions the graph shape cannot: **full-text** over string
properties, and **vector** over embeddings.
Both answer questions the graph shape cannot: *which questions mention X* and
*which questions mean the same as this one*.

A full-text index over the titles turns a term into ranked matches — the
`SEARCH … IN (FULLTEXT INDEX …)` subclause (GQL's index-search syntax) returns
the score alongside the row, so it composes with any `RETURN`:

In [12]:
run("create a full-text index over question titles", """
    CREATE FULLTEXT INDEX question_titles FOR (q:Question) ON EACH [q.title]
    """)

run("full-text search for the tag name", """
    MATCH (q:Question)
    SEARCH q IN (FULLTEXT INDEX question_titles FOR $term LIMIT 5) SCORE AS score
    RETURN q.title, q.score AS question_score, score
    ORDER BY score DESC
    """, {"term": TAG})


create a full-text index over question titles (0 rows, 1.2 ms)
  changed: 1 indexes added

full-text search for the tag name (5 rows, 0.7 ms)
  q.title                                                        question_score  score              
  -------------------------------------------------------------  --------------  -------------------
  Neo4j CALL {} IN TRANSACTIONS Does Not Work in Neo4j HTTP API  0               0.11278036597801411
  Neo4j Reactive Session issue                                   1               0.11052305272671194
  Understanding indexes in Neo4j                                 0               0.11052305272671194
  Neo4j vector similarity function                               1               0.11052305272671194
  neo4j cluster HA setup                                         0               0.11052305272671194


For vector search, every question needs an embedding first. A real
application would embed with a model (say `sentence-transformers`); the
notebook stays dependency-free and offline with a tiny deterministic
embedding: hash each token into one of 64 dimensions, add its weight, and
L2-normalize — cosine similarity then compares the token profiles.

In [13]:
import hashlib
import math

EMBEDDING_DIM = 64


def embed(text):
    """A deterministic 64-dim hashing embedding: token -> dimension, L2-normalized."""
    vec = [0.0] * EMBEDDING_DIM
    tokens = [tok for tok in text.lower().replace("/", " ").split() if tok]
    for tok in tokens:
        digest = hashlib.blake2b(tok.encode(), digest_size=4).digest()
        vec[int.from_bytes(digest, "little") % EMBEDDING_DIM] += 1.0
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


# Embed every title and store the vector as a property.
rows = db.execute(
    "MATCH (q:Question) RETURN q.uuid AS uuid, q.title AS title"
).fetchall()
embedded = [
    {"uuid": r["uuid"], "embedding": embed(r["title"] or "")} for r in rows
]
db.execute("UNWIND $rows AS row MATCH (q:Question {uuid: row.uuid}) SET q.embedding = row.embedding",
           {"rows": embedded})
print(f"embedded {len(embedded)} titles into {EMBEDDING_DIM} dimensions")

embedded 100 titles into 64 dimensions


In [14]:
# `WITH [...]` lists the properties the SEARCH clause may filter on: they are
# carried inside the index, so a filtered vector search never touches the graph.
run("create a cosine vector index over the embeddings", """
    CREATE VECTOR INDEX question_embeddings FOR (q:Question) ON (q.embedding)
    WITH [q.is_answered, q.up_vote_count, q.creation_date]
    OPTIONS {indexConfig: {`vector.dimensions`: 64,
                          `vector.similarity_function`: 'cosine'}}
    """)

query = "graph database query language"
run("nearest questions to a phrase", """
    MATCH (q:Question)
    SEARCH q IN (VECTOR INDEX question_embeddings FOR $vec LIMIT 10) SCORE AS score
    RETURN q.title, score
    ORDER BY score DESC LIMIT 5
    """, {"vec": embed(query)})

# The index-carried properties filter inside the search: only answered,
# upvoted questions come back, ranked by similarity.
run("nearest answered questions", """
    MATCH (q:Question)
    SEARCH q IN (VECTOR INDEX question_embeddings FOR $vec LIMIT 100) SCORE AS score
    WHERE q.is_answered = true AND q.creation_date >= $since
    RETURN q.title, q.creation_date, score
    ORDER BY score DESC LIMIT 5
    """, {"vec": embed(query), "since": int(time.time()) - 365 * 24 * 3600})


create a cosine vector index over the embeddings (0 rows, 0.4 ms)
  changed: 1 indexes added

nearest questions to a phrase (5 rows, 0.2 ms)
  q.title                                                                                     score             
  ------------------------------------------------------------------------------------------  ------------------
  Neo4j has issues with loading Reactome Graph Database                                       0.7651650309562683
  Neo4j graph creation query                                                                  0.75              
  Unable to run neo4j create constraint cypher query from Databricks using pyspark connector  0.7425355911254883
  Unable to access neo4j db when URL is made https                                            0.7165063619613647
  Best way to exclude `null` or empty [] values from an output of a Cypher query              0.6875            

nearest answered questions (4 rows, 0.7 ms)
  q.title             

Both index kinds show up in `SHOW INDEXES`, next to the range indexes:

In [15]:
run("all indexes", """
    SHOW INDEXES YIELD name, type, labelsOrTypes, properties
    RETURN name, type, labelsOrTypes, properties
    ORDER BY type, name
    """)


all indexes (7 rows, 0.4 ms)
  name                 type      labelsOrTypes  properties                                                    
  -------------------  --------  -------------  --------------------------------------------------------------
  question_titles      FULLTEXT  ['Question']   ['title']                                                     
  index_1b9dcc97       LOOKUP    None           None                                                          
  index_460996c0       LOOKUP    None           None                                                          
  constraint_233a8716  RANGE     ['Question']   ['uuid']                                                      
  constraint_83d65333  RANGE     ['Tag']        ['name']                                                      
  constraint_c67c3cd5  RANGE     ['Answer']     ['uuid']                                                      
  question_embeddings  VECTOR    ['Question']   ['embedding', 'is_answered', 'up_v

---
## 7. A look at the loaded data
The same helper renders a real slice of the graph: the 20 newest questions with
their tags, askers, and answers, colored by label. Captions are per label — the
question's title, the tag's name, the user's display name, and whether an answer
is accepted — mapped explicitly, because the label alone says nothing.

Static preview:

![data sample](assets/stackoverflow_sample.png)

In [16]:
SAMPLE = 20  # questions in the data sample

# Per-label captions for the data views.
CAPTIONS = {
    "Question": "title",
    "Tag": "name",
    "User": "display_name",
    "Answer": "is_accepted",
}

# Per-label sizing values for the data views: a question scales with its
# score (area-proportional, clamped at 0); everything else is small + fixed.


def sample_value(n):
    """Sizing value per label: a question's score, a small constant otherwise."""
    return n.properties["score"] if n.properties["labels"] == ["Question"] else 1

# Return the graph elements themselves (paths): relate.viz.to_visualization_graph
# picks up every node and relationship they touch.
sample = relate.viz.to_visualization_graph(
    db.execute(
        """
        MATCH (q:Question) WITH q ORDER BY q.creation_date DESC LIMIT $sample
        MATCH p=(q)-[r]-(n)
        RETURN p
        """,
        {"sample": SAMPLE},
    ),
    node_caption=CAPTIONS,
)

size_nodes(sample, sample_value)
relate.viz.render_html(sample, initial_zoom=1)

---
## 8. Basic queries
Labels and relationship types in the graph, with counts — the first exploration
queries of the Stack Overflow example.

In [17]:
run("labels", """
    MATCH (n)
    RETURN labels(n) AS label, count(*) AS freq
    ORDER BY freq DESC
    """)

run("relationship types", """
    MATCH ()-[r]->()
    RETURN type(r) AS type, count(*) AS freq
    ORDER BY freq DESC
    """)


labels (4 rows, 0.1 ms)
  label         freq
  ------------  ----
  ['User']      122 
  ['Question']  100 
  ['Answer']    99  
  ['Tag']       96  

relationship types (4 rows, 0.3 ms)
  type      freq
  --------  ----
  TAGGED    293 
  ASKED     100 
  PROVIDED  99  
  ANSWERED  99  


---
## 9. Top tags and users
The most popular tags, who is asking, and who is answering.

In [18]:
run("top tags", """
    MATCH (q:Question)-[:TAGGED]->(t:Tag)
    RETURN t.name AS tag, count(q) AS questions
    ORDER BY questions DESC LIMIT 5
    """)

run("top askers", """
    MATCH (u:User)-[:ASKED]->(q:Question)
    RETURN u.display_name AS user, count(*) AS questions
    ORDER BY questions DESC LIMIT 10
    """)

run("top answerers", """
    MATCH (u:User)-[:PROVIDED]->(a:Answer)-[:ANSWERED]->(q:Question)
    RETURN u.display_name AS user, count(a) AS answers, avg(a.score) AS avg_score
    ORDER BY answers DESC LIMIT 10
    """)


top tags (5 rows, 0.2 ms)
  tag     questions
  ------  ---------
  neo4j   100      
  cypher  38       
  python  9        
  graph   8        
  java    8        

top askers (10 rows, 0.1 ms)
  user                questions
  ------------------  ---------
  it_would_be_better  5        
  norihide.shimatani  3        
  AFatNiBBa           2        
  Kt Student          2        
  Martin              2        
  goubint             2        
  filippo             2        
  py-r                2        
  Aravind_Sagar       2        
  BairDev             1        

top answerers (10 rows, 0.5 ms)
  user                  answers  avg_score         
  --------------------  -------  ------------------
  cybersam              29       0.7241379310344828
  Finbar Good           12       1.5               
  Christoffer Bergman   6        0.8333333333333334
  meistermeier          4        0.75              
  Christophe Willemsen  4        0.25              
  Therese             

---
## 10. Unanswered questions
Tags of questions with no answers (excluding the seed tag itself), so you can see
where questions are going unanswered.

In [19]:
run("tags of unanswered questions", """
    MATCH (q:Question)-[:TAGGED]->(t:Tag)
    WHERE NOT t.name IN [$tag] AND NOT (q)<-[:ANSWERED]-()
    RETURN t.name AS tag, count(q) AS questions
    ORDER BY questions DESC LIMIT 10
    """, {"tag": TAG})


tags of unanswered questions (10 rows, 0.4 ms)
  tag                questions
  -----------------  ---------
  cypher             8        
  python             2        
  graph              2        
  spring-boot        2        
  graphql            1        
  database           1        
  nodes              1        
  best-practices     1        
  java               1        
  spring-data-neo4j  1        


---
## 11. Tag co-occurrence and similarity (TagOverflow)
Which tags appear together on the same questions? The [TagOverflow]
(https://medium.com/data-science/tagoverflow-correlating-tags-in-stackoverflow-66e2b0e1117b)
article starts from the raw **co-occurrence count** — the number of questions
carrying both tags — and then ranks by **Jaccard similarity**, the same count
normalized by how much either tag covers:

```text
count(A, B)   = |questions(A) ∩ questions(B)|
jaccard(A, B) = |questions(A) ∩ questions(B)| / |questions(A) ∪ questions(B)|
```

The count favors big tags (two huge tags share questions just by size); Jaccard
is comparable across tags of any size. Both come from one computation and land
on one relationship — `CO_OCCURS_WITH {count, jaccard}` — so the two rankings
can be read side by side and can never drift apart. Section 12 draws the result.

In [20]:
# Pairs are only interesting once two tags share more than one question:
# a single shared question gives every obscure tag pair a perfect Jaccard score.
MIN_COOCCURRENCE = 2

# One statement computes both metrics per pair (count == inter).
cursor = db.execute("""
    MATCH (t1:Tag)<-[:TAGGED]-(q1:Question)
    WITH t1, collect(id(q1)) AS qs1
    MATCH (t2:Tag)<-[:TAGGED]-(q2:Question)
    WHERE id(t1) < id(t2)
    WITH t1, t2, qs1, collect(id(q2)) AS qs2
    WITH t1, t2, size(qs1) AS d1, size(qs2) AS d2,
         [x IN qs1 WHERE x IN qs2] AS both
    WITH t1, t2, d1, d2, size(both) AS inter
    WHERE inter >= $min
    RETURN t1.name AS tag1, t2.name AS tag2, inter AS count, d1, d2,
           (inter * 1.0) / (d1 + d2 - inter) AS jaccard
    """, {"min": MIN_COOCCURRENCE})
pairs = cursor.fetchall()
elapsed_ms = cursor.consume().elapsed_ms

# The two rankings disagree, and that is the point: by count, the biggest
# tags win; by Jaccard, tightly related niches surface.
show("tag pairs by co-occurrence count",
     sorted(pairs, key=lambda r: r["count"], reverse=True)[:10],
     elapsed_ms, cursor.stats)
show("tag pairs by Jaccard similarity",
     sorted(pairs, key=lambda r: r["jaccard"], reverse=True)[:10],
     elapsed_ms, cursor.stats)

db.execute(
    """
    UNWIND $pairs AS p
    MATCH (a:Tag {name: p.tag1}), (b:Tag {name: p.tag2})
    MERGE (a)-[r:CO_OCCURS_WITH]->(b)
      SET r.count = p.count, r.jaccard = p.jaccard
    """,
    {"pairs": pairs},
)
print(f"{len(pairs)} CO_OCCURS_WITH relationships materialized")

# Tag question counts, used to size the tags in section 12.
tag_questions = {
    r["name"]: r["questions"]
    for r in db.execute("""
        MATCH (t:Tag)
        OPTIONAL MATCH (q:Question)-[:TAGGED]->(t)
        RETURN t.name AS name, count(q) AS questions
        """).fetchall()
}
tag_size = lambda n: tag_questions.get(n.properties["name"], 1)


tag pairs by co-occurrence count (10 rows, 25.7 ms)
  tag1    tag2               count  d1   d2  jaccard
  ------  -----------------  -----  ---  --  -------
  neo4j   cypher             38     100  38  0.38   
  neo4j   python             9      100  9   0.09   
  neo4j   graph              8      100  8   0.08   
  neo4j   java               8      100  8   0.08   
  neo4j   spring-data-neo4j  7      100  7   0.07   
  neo4j   spring-boot        6      100  6   0.06   
  cypher  graph              6      38   8   0.15   
  neo4j   database           5      100  5   0.05   
  neo4j   neo4j-apoc         4      100  4   0.04   
  neo4j   knowledge-graph    3      100  3   0.03   

tag pairs by Jaccard similarity (10 rows, 25.7 ms)
  tag1               tag2                  count  d1   d2  jaccard            
  -----------------  --------------------  -----  ---  --  -------------------
  apache-kafka       apache-kafka-connect  2      2    2   1.0                
  neo4j              c

---
## 12. Visualize the tag graph
Return the similar tags with their relationships — `relate.viz` maps the
query's graph elements onto [neo4j-viz]
(https://neo4j.com/docs/python-graph-visualization/current/) entities directly:
node captions come from the labels, edge captions from the type, and node colors
from the labels via the Neo4j palette. Node size carries the question count, and
edges carry both numbers: sized and labeled with the count, with the Jaccard
score in the edge caption.

All four visualizations render with `relate.viz.render_html` — neo4j-viz's
standalone HTML, but with the graph data block's id made unique per call, so each
cell paints its own graph (the template otherwise looks the data up by a
page-global id and every cell paints the first one). In JupyterLab, Notebook 7,
or VS Code (fixed in neo4j-viz 1.11.0) you can instead render the same
`VisualizationGraph` interactively with its `render_widget()`.

Static preview:

![tag similarity](assets/stackoverflow_similarity.png)

In [21]:
VG = relate.viz.to_visualization_graph(
    db.execute("""
        MATCH (a:Tag)-[r:CO_OCCURS_WITH]->(b:Tag)
        RETURN a, r, b
        """),
    node_caption=CAPTIONS,
)

print(f"{len(VG.nodes)} tags, {len(VG.relationships)} co-occurrence edges")

# Size the tags by their question count (`size_nodes` is defined in section 5,
# `tag_size` in section 11) and caption the edges with the Jaccard score.
size_nodes(VG, tag_size)
for r in VG.relationships:
    r.caption = f'{r.properties["count"]} / {r.properties["jaccard"]:.2f}'

relate.viz.render_html(VG, initial_zoom=1)

28 tags, 40 co-occurrence edges


---
## 13. Cleanup
The database is in-memory; close it to release the graph.

In [22]:
db.close()
print(f"Connection closed: {db.closed}")

Connection closed: True
